# Activity 21.1: Reinforcement Learning Activity

Learning Outcomes Addressed:
- 5. Implement the Quality matrix and the Bellman equation.
- 6. Implement the fundamental steps of reinforcement learning.


## Activity Overview

In this activity, you will explore how implementing an algorithm using reinforcement learning can improve the performance of your results. In particular, you will work with a Python implementation that simulates the movements of a self-driving cab to calculate how many steps it takes to pick up and drop off a passenger at the correct locations.

You will consider the implementations of this problem with and without reinforcement learning and you will be asked to compare the performance, results, and penalties of the algorithms.

## Problem Description

Imagine that you want to design a simulation for a self-driving cab (the agent) that wants to pick up a passenger in a certain location and safely drop them off at another one by taking the minimum amount of steps possible.

Before you dive deeper into how to implement this problem without and with reinforcement learning, you need to consider and define the following: rewards, states, and actions.

#### Rewards

- The agent should receive a high positive reward if he drops off the passenger successfully.
- The agent should be penalized if it tries to drop off a passenger in the wrong locations.
- The agent should get a small negative reward for not making it to the destination after every time-step. In other words, you want to penalize the agent a little if he doesn't make the optimal move.

#### State Space

Assume that you model the parking lot for this activity as a $5 \times 5$ grid, which gives you 25 possible taxi locations. Assume further that you can pick up or drop off a passenger at four different locations $(0,0), (0,4), (4,0), (4,3)$. 

Also, assume that you want to pick-up your passenger at point $S$ and drop him off at point $F$.

Take into account one additional state which describes the passenger as being inside of the cab.

Therefore, given the dimension of the grid, the total number of possible destinations, and the possible states for the passenger you have a total of $5 \times 5\times 4 \times 5 = 500$ states.

#### Action Space

Your agent has six possible actions:

- Moving South
- Moving North
- Moving East
- Moving West
- Picking up the passenger
- Dropping off the passenger

These six actions define the action space for your agent. Given the map above, you will see that, due to the presence of walls, your agent won't be able to perform some actions. In your code, you will define these walls by providing a -1 penalty for every hit.

## Part 1: Problem Setup

The problem described above may seem quite challenging to implement in Python.

Luckily, the OpenAi [`gym` *library*](https://gym.openai.com/) already offers an environment that does exactly what you want.

In the code cell below, import the *library* and invoke the `"Taxi-v3"` environment that already has all the rewards, state spaces, and actions as defined above. You will also set a random seed for reproducibility of the results.

Run the code cell below to import and render the environment.


In [ ]:
import gym, time

env = gym.make("Taxi-v3", render_mode="human") 
env.reset(seed=0)
for _ in range(1000000):
    env.step(env.action_space.sample()) 
    time.sleep(1)                       
env.render()

In the visualization above, the agent (the self-driving cab) is represented by the yellow rectangle. The agent will turn green once the passenger has been picked up.

You can also observe that there are four possible pick-up and drop-off locations represented by the letters R (red), G (green), Y (yellow), and B (blue). The code automates the to blue letter represent the passenger pick-up location, and the purple letter to represent the destination.

Finally, the pipe ("|") represents a wall which the self-driving cab cannot cross.

Next, you want to verify that your environment is defined correctly.

The code cell below will call the `action_space` *method* to *print* the total number of actions in your problem.

Run the code cell below.

In [4]:
print("Action Space {}".format(env.action_space))


Action Space Discrete(6)


## Question 1

Similarly to the previous code cell, complete the *print* *statement* by calling the `observation_space` *method* to *print* the number of states in your environment.

In [ ]:
print("State Space {}".format(   ))

IndexError: Replacement index 0 out of range for positional args tuple

: 

Now you can see that your environment has necessary 500 states.

## Question 2

Next, you need to encode the position of your agent and the pick-up and drop-off locations in your environment.

Observe the visualization of your environment above.

In the code cell below, fill in the ellipsis, keeping the following in mind:
- The first entry represents the agent row *index*, where each row is numbered from 0 to 4 from top to bottom.
- The second entry represents the agent column *index*, where each column is numbered from 0 to 4 from left to right.
- The third entry represents the passenger pick-up location. The passenger pick-up locations are encoded in the following way:
   - 0: R (red)
   - 1: G (green)
   - 2: Y (yellow)
   - 3: B (blue)
   - 4: in taxi
- The fourth entry represents the passenger drop-off location. Encode that in a similar way as the pick-up location.

In [ ]:
# (taxi row, taxi column, passenger index, destination index)
state = env.encode(..., ..., ..., ...) 


Run the code below to generate a number that corresponds to a `state` between 0 and 499.

In [ ]:
print("State:", state)

You can observe that the original environment corresponds to `state` 26.


## The Reward Matrix

When you created your environment, the *library* also created an initial reward matrix that has the number of states represented as rows and number of actions represented as columns.

## Question 3

In the code cell below, fill in the ellipsis with the `state` generated in the previous cell.

In [ ]:
env.P[...]

The *dictionary* above has the following structure: *`{action: [(probability, nextstate, reward, done)]}`*.

A few things to note:

- The 0-5 corresponds to the actions (South, North, East, West, pick-up, drop-off) that the agent can perform at your current state in the illustration.
- In this environment, `probability` is always 1.0.
- The `nextstate` is the state you would be in if you take the action at this *index* of the *dictionary*.
- All the movement actions have a `reward` of -1. The pick-up and drop-off actions have a `reward` of -10 in this particular state. If you are in a state where the agent has a passenger and is at the correct destination, you would see a `reward` of 20 at the drop-off action.
- The `done` result is used to tell you when you have successfully dropped off a passenger in the correct location.

Note that if your agent chooses to explore action two (2) in this state, it would be going East into a wall. The source code has made it impossible to actually move the taxi across a wall, so if the agent chooses that action, it will just keep accumulating -1 penalties, which affects the long-term reward.

## Part 2: Python Implementation without Reinforcement Learning

In the code cell below, a solution has been implemented to pick up and drop off the passenger without reinforcement learning.

Note that you start with the same state (26) that was determined previously.

## Question 4

Observe the code in the code cell below. What is the condition that will make the infinite `while` *loop* end? This is an open-ended question that requires a written response.

Question 4: (Write your answer here.)

Next, run the code cell below. Notice that this is a random problem, the number of steps and the number of penalties will change each time you run the code cell below.

In [ ]:
env.s = 26  # set environment to illustration's state

epochs = 0
penalties, reward = 0, 0

frames = [] # for animation

done = False

while not done:
    action = env.action_space.sample()
    state, reward, done, info = env.step(action)

    if reward == -10:
        penalties += 1
    
    # Put each rendered frame into dict for animation
    frames.append({
        'frame': env.render(mode='ansi'),
        'state': state,
        'action': action,
        'reward': reward
        }
    )

    epochs += 1
    
    
print("Steps taken: {}".format(epochs))
print("Penalties incurred: {}".format(penalties))

Run the code cell below to see your agent in action.

Note that this cell will display an number of frames equal to the number of steps returned by the previous code cell.
Feel free to stop the code cell below to run; it won't impact the results of your algorithm.

In [ ]:
from IPython import display
from time import sleep

env.reset()
def print_frames(frames):
    for i, frame in enumerate(frames):
    

        new_state, reward, done, info = env.step(env.action_space.sample())

    # Display code
        display.clear_output(wait=True)
        display.display(env.render())
        print(f"Timestep: {i + 1}")
        print(f"State: {frame['state']}")
        print(f"Action: {frame['action']}")
        print(f"Reward: {frame['reward']}")
        sleep(1)
        
print_frames(frames)        

## Part 3: Python Implementation with Reinforcement Learning

In the last part of this activity, you will use reinforcement learning and the Quality matrix to optimize your algorithm.

## Question 5

Define a Quality matrix with all zero entries and dimensions $500 \times 6$. Assign this matrix to the `q_table` variable.

In [ ]:
import numpy as np
q_table = np.zeros(   )

You can now create the training algorithm that will update this Quality matrix as the agent explores the environment over thousands of episodes.


## Question 6

Observe the code in the code cell below. What is the main difference between this implementation and the one proposed in Part 2? Which equation is used to train the algorithm and to fill the entries of the Quality matrix? This is an open-ended question that requires a written response.

Question 6: (Write your answer here.)

Run the code cell below to train your agent.

In [ ]:
%%time
"""Training the agent"""
import numpy as np
import random
from IPython.display import clear_output

# Hyperparameters
alpha = 0.1
gamma = 0.6
epsilon = 0.1

# For plotting metrics
all_epochs = []
all_penalties = []

for i in range(1, 10001):
    state = env.reset()

    epochs, penalties, reward, = 0, 0, 0
    done = False
    
    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore action space
        else:
            action = np.argmax(q_table[state]) # Exploit learned values

        next_state, reward, done, info = env.step(action) 
        
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
        
        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state, action] = new_value

        if reward == -10:
            penalties += 1

        state = next_state
        epochs += 1
        
    if i % 10 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")

print("Training finished.\n")

You can now evaluate the performance of your agent. 

Notice that now the next action is always selected using the best value from the Quality matrix.

Run the code cell below to see how your algorithm performs after picking up and dropping off 10 passengers.

In [ ]:
"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
passengers = 10

for _ in range(passengers):
    state = env.reset()
    epochs, penalties, reward = 0, 0, 0
    
    done = False
    
    while not done:
        action = np.argmax(q_table[state])# next action selected using the best value in the Quality matrix
        state, reward, done, info = env.step(action)

        if reward == -10:
            penalties += 1

        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {passengers} episodes:")
print(f"Average steps per episode: {total_epochs / passengers}")
print(f"Average penalties per episode: {total_penalties / passengers}")


## Question 7

Compare the performances of the two approaches presented in this activity. Which one performs better? In your summary, include the number of steps taken each time the algorithm ran and the penalties that were incurred. This is an open-ended question that requires a written response.

Question 7: (Write your answer here.)